# Numcompute Stream Demo

This notebook demonstrates the streaming capabilities of `numcompute_stream`.

The demo will go through the following:

1. Load a student-performance CSV file through the custom `io.py` loader.
2. Split the records into chunks to simulate data arriving over time.
3. Build two incremental pipelines: a decision tree and a bagging ensemble.
4. Use **predict-then-learn** evaluation for each new chunk.
5. Log accuracy and memory footprint through `StreamTrainer`.
6. Visualise accuracy, error rate, model comparison, memory usage, and the latest predictions through `visualise.py`.

## 1. Import the custom library

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import time
from pathlib import Path

import numpy as np
from numcompute_stream.io import load_csv
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import SimpleImputer, MinMaxScaler, StandardScaler, OneHotEncoder
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.visualise import plot_metric_over_time, compare_models, plot_predictions_vs_ground_truth

## 2. Load the dataset via custom I/O pipeline

`sample_student_performance.csv` has the following structure:

| Column | Type | Meaning |
|---|---|---|
| `student_id` | Identifier | Excluded from model training |
| `study_hours_week` | Numeric | Weekly study hours |
| `attendance_pct` | Numeric | Attendance percentage |
| `assignment_avg` | Numeric | Average assignment score |
| `previous_exam_score` | Numeric | Previous exam score |
| `sleep_hours` | Numeric | Average nightly sleep |
| `school_type` | Categorical | `0 = public`, `1 = private` |
| `study_group` | Categorical | `0 = self-study`, `1 = peer`, `2 = tutoring` |
| `internet_access` | Categorical | `0 = no`, `1 = yes` |
| `passed` | Target | `0 = did not pass`, `1 = passed` |

In [3]:
# Load data
file_name = Path.cwd().parent / "benchmark/sample_student_performance.csv"
print(file_name)
student_data = load_csv(file_name, delimiter=",", dtype=float, skip_header=True)

# Variables preparation
X = student_data[:, 1:-1]                  # Exclude student_id and target
y = student_data[:, -1].astype(int)         # Binary target (0, 1)

NUMERIC_INDEXES = np.array([0, 1, 2, 3, 4])
CATEGORICAL_INDEXES = np.array([5, 6, 7])

print('Loaded shape:', student_data.shape)
print('First three rows:\n', student_data[:3])

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('Missing numeric values:', int(np.isnan(X[:, NUMERIC_INDEXES]).sum()))
print('Class counts:', np.unique(y, return_counts=True))

/Users/vaifathuy/Documents/AU/COMP5004 - Python For ML/Projects/assignment2.2/benchmark/sample_student_performance.csv
Loaded shape: (80, 10)
First three rows:
 [[1.001e+03 1.750e+01 8.590e+01 5.070e+01 8.680e+01 8.500e+00 0.000e+00
  0.000e+00 0.000e+00 1.000e+00]
 [1.002e+03 1.080e+01 7.510e+01 9.350e+01 8.500e+01 6.800e+00 1.000e+00
  1.000e+00 1.000e+00 1.000e+00]
 [1.003e+03 1.920e+01 9.220e+01 5.720e+01 5.460e+01 6.200e+00 0.000e+00
  2.000e+00 1.000e+00 1.000e+00]]
Feature matrix shape: (80, 8)
Target shape: (80,)
Missing numeric values: 10
Class counts: (array([0, 1]), array([40, 40]))


## 3. Split the dataset into chunks

The complete CSV is split into eight chunks of ten records each. The model receives one chunk at a time.

In [4]:
CHUNK_SIZE = 10
X_chunks = [X[i:i + CHUNK_SIZE] for i in range(0, len(X), CHUNK_SIZE)]
y_chunks = [y[i:i + CHUNK_SIZE] for i in range(0, len(y), CHUNK_SIZE)]

print('Number of chunks:', len(X_chunks))
print('Rows per chunk:', [len(chunk) for chunk in X_chunks])

Number of chunks: 8
Rows per chunk: [10, 10, 10, 10, 10, 10, 10, 10]


In [5]:
category_names = ['school_type', 'study_group', 'internet_access']

for name, relative_index in zip(category_names, range(len(CATEGORICAL_INDEXES))):
    feature_values = np.unique(X[:, CATEGORICAL_INDEXES[relative_index]])
    initial_values = np.unique(X_chunks[0][:, CATEGORICAL_INDEXES[relative_index]])
    print(f'{name}: initial={initial_values}, complete={feature_values}')
    assert np.array_equal(initial_values, feature_values)

print('All categorical levels appear in the first chunk.')

school_type: initial=[0. 1.], complete=[0. 1.]
study_group: initial=[0. 1. 2.], complete=[0. 1. 2.]
internet_access: initial=[0. 1.], complete=[0. 1.]
All categorical levels appear in the first chunk.


## 4. Create preprocessor coordinator and streaming pipelines

### Custom prepressors for numerical and categorical

The custom preprocessors operate on arrays, but numeric and categorical columns require different treatment. This small adapter applies each uploaded preprocessor to the appropriate subset of columns and combines the outputs. 

In [6]:
class StudentFeaturePreprocessor:
    def __init__(self, numeric_indexes, categorical_indexes):
        self.numeric_indexes = np.asarray(numeric_indexes)
        self.categorical_indexes = np.asarray(categorical_indexes)
        self.imputer = SimpleImputer(strategy='mean', fill_value=0.0)
        self.minmax_scaler = MinMaxScaler()
        self.standard_scaler = StandardScaler()
        self.encoder = OneHotEncoder(handle_unknown='ignore')

    def fit(self, X, y=None):
        self.__init__(self.numeric_indexes, self.categorical_indexes)
        return self.partial_fit(X, y)

    def partial_fit(self, X, y=None):
        numeric = X[:, self.numeric_indexes]
        categorical = X[:, self.categorical_indexes]

        self.imputer.partial_fit(numeric, y)
        numeric_clean = self.imputer.transform(numeric).astype(float)

        self.minmax_scaler.partial_fit(numeric_clean, y)
        self.standard_scaler.partial_fit(numeric_clean, y)
        self.encoder.partial_fit(categorical, y)

        return self

    def transform(self, X):
        numeric = X[:, self.numeric_indexes]
        categorical = X[:, self.categorical_indexes]

        numeric_clean = self.imputer.transform(numeric).astype(float)
        numeric_minmax = self.minmax_scaler.transform(numeric_clean)
        numeric_standard = self.standard_scaler.transform(numeric_clean)
        categorical_encoded = self.encoder.transform(categorical)

        return np.hstack([
            numeric_minmax,
            numeric_standard,
            categorical_encoded,
        ])

### Build two incremental pipelines

Both pipelines use the same feature adapter `StudentFeaturePreprocessor`. The first pipeline ends with one decision tree. The second ends with a bagging ensemble of trees.

In [7]:
def build_tree_pipeline():
    return Pipeline([
        ('features', StudentFeaturePreprocessor(NUMERIC_INDEXES, CATEGORICAL_INDEXES)),
        ('model', DecisionTreeClassifier(
            max_depth=4,
            min_samples_split=4,
            random_state=42,
        )),
    ])

def build_ensemble_pipeline():
    return Pipeline([
        ('features', StudentFeaturePreprocessor(NUMERIC_INDEXES, CATEGORICAL_INDEXES)),
        ('model', EnsembleClassifier(
            n_estimators=7,
            max_depth=4,
            min_samples_split=4,
            random_state=42,
        )),
    ])

tree_trainer = StreamTrainer(build_tree_pipeline())
ensemble_trainer = StreamTrainer(build_ensemble_pipeline())

## 5. Train incrementally with `.partial_fit()`

The first chunk is used to initialise each model. For every later chunk, it follows a realistic **predict-then-learn** order to prevent the model from being evaluated on data it has already learned from.

In [8]:
# Initialise both pipelines with the first chunk
tree_trainer.fit_chunk(X_chunks[0], y_chunks[0])
ensemble_trainer.fit_chunk(X_chunks[0], y_chunks[0])

latest_tree_predictions = None
latest_ensemble_predictions = None
latest_y_true = None

print(f"{'Chunk':<8}{'Tree Accuracy':<18}{'Ensemble Accuracy':<20}")
print("-" * (8 + 18 + 20))

for chunk_index in range(1, len(X_chunks)):
    X_chunk = X_chunks[chunk_index]
    y_chunk = y_chunks[chunk_index]

    # Evaluate before learning from this chunk.
    tree_accuracy = tree_trainer.score_chunk(X_chunk, y_chunk)
    ensemble_accuracy = ensemble_trainer.score_chunk(X_chunk, y_chunk)

    latest_tree_predictions = tree_trainer.model.predict(X_chunk)
    latest_ensemble_predictions = ensemble_trainer.model.predict(X_chunk)
    latest_y_true = y_chunk.copy()

    print(
        f"{chunk_index + 1:<8}"
        f"{tree_accuracy:<18.2f}"
        f"{ensemble_accuracy:<20.2f}"
    )

    # Incrementally update both pipelines.
    tree_trainer.fit_chunk(X_chunk, y_chunk)
    ensemble_trainer.fit_chunk(X_chunk, y_chunk)

    time.sleep(0.5)

print()
print('Summary:')
print('Tree fit chunks:', tree_trainer.fit_chunk_count)
print('Ensemble fit chunks:', ensemble_trainer.fit_chunk_count)
print('Evaluated chunks:', ensemble_trainer.score_chunk_count)

Chunk   Tree Accuracy     Ensemble Accuracy   
----------------------------------------------
2       0.70              0.70                
3       0.70              0.80                
4       0.80              0.80                
5       0.90              0.80                
6       0.70              0.50                
7       0.40              0.60                
8       0.60              0.80                

Summary:
Tree fit chunks: 8
Ensemble fit chunks: 8
Evaluated chunks: 7


## 6. Inspect the logs

`StreamTrainer` records per-chunk accuracy, cumulative accuracy, and traced Python-memory usage. The table below keeps the output lightweight for a screen-recording demo.

In [ ]:
tree_logs = tree_trainer.logs
ensemble_logs = ensemble_trainer.logs

print(
    f"{'Scored Chunk':<14}"
    f"{'Tree Acc.':<12}"
    f"{'Tree Cum.':<12}"
    f"{'Tree KB':<14}"
    f"{'Ensemble Acc.':<16}"
    f"{'Ensemble Cum.':<16}"
    f"{'Ensemble KB':<14}"
)
print("-" * 98)

for tree_log, ensemble_log in zip(tree_logs, ensemble_logs):
    print(
        f"{ensemble_log['chunk']:<14}"
        f"{tree_log['chunk_accuracy']:<12.2f}"
        f"{tree_log['cumulative_accuracy']:<12.2f}"
        f"{tree_log['memory_bytes'] / 1024:<14.2f}"
        f"{ensemble_log['chunk_accuracy']:<16.2f}"
        f"{ensemble_log['cumulative_accuracy']:<16.2f}"
        f"{ensemble_log['memory_bytes'] / 1024:<14.2f}"
    )

print()
print("Cumulative Accuracy Summary")
print(f"Tree: {tree_trainer.cumulative_accuracy:.2f}")
print(f"Ensemble: {ensemble_trainer.cumulative_accuracy:.2f}")

## 7. Visualise cumulative accuracy over time

This chart is created by the reusable `plot_metric_over_time()` function from `visualise.py`.

In [ ]:
# Prepare data for plotting
tree_chunk_accuracy = np.array([
    log["chunk_accuracy"]
    for log in tree_logs
])

tree_cumulative_accuracy = np.array([
    log["cumulative_accuracy"]
    for log in tree_logs
])

ensemble_chunk_accuracy = np.array([
    log["chunk_accuracy"]
    for log in ensemble_logs
])

ensemble_cumulative_accuracy = np.array([
    log["cumulative_accuracy"]
    for log in ensemble_logs
])

# Plotting
plot_metric_over_time(
    metric_values=tree_cumulative_accuracy,
    title="Tree: Cumulative Accuracy over Time",
    ylabel="Cumulative Accuracy",
)

plot_metric_over_time(
    metric_values=ensemble_cumulative_accuracy,
    title="Ensemble: Cumulative Accuracy over Time",
    ylabel="Cumulative Accuracy",
)


## 8. Visualise the chunk-by-chunk error rate

The plots below shows error rate of chunk-based prediction of `Tree` vs `Ensemble`.

In [ ]:
# Chunk-by-chunk error rates
tree_error_rate = 1 - tree_chunk_accuracy
ensemble_error_rate = 1 - ensemble_chunk_accuracy

compare_models(
    metric1=tree_error_rate,
    metric2=ensemble_error_rate,
    labels=[
        "Decision Tree",
        "Bagging Ensemble"
    ],
    title="Chunk Error Rate: Decision Tree vs Ensemble",
    ylabel="Error Rate"
)


## 9. Compare the two models `Tree` vs `Ensemble`

The custom `compare_models()` helper plots the cumulative accuracy of the single tree and the bagging ensemble on the same axes.

In [ ]:
compare_models(
    metric1=tree_cumulative_accuracy,
    metric2=ensemble_cumulative_accuracy,
    labels=(
        "Decision Tree",
        "Bagging Ensemble",
    ),
    title="Streaming Model Comparison",
    ylabel="Cumulative Accuracy",
)

## 10. Visualise predictions for the latest evaluated chunk

The final plot compares the ensemble predictions with the actual pass/fail outcomes before the latest chunk was learned.

In [ ]:
plot_predictions_vs_ground_truth(
    y_true=latest_y_true,
    y_pred=latest_ensemble_predictions,
    title="Latest Chunk: Ensemble Predictions vs Ground Truth",
)

## 11. Visual traced memory usage

In [ ]:
tree_memory_kb = np.array([
    log["memory_bytes"] / 1024
    for log in tree_logs
])

ensemble_memory_kb = np.array([
    log["memory_bytes"] / 1024
    for log in ensemble_logs
])

plot_metric_over_time(
    metric_values=tree_memory_kb,
    title="Tree Traced Memory Usage over Time",
    ylabel="Memory (KB)",
)

plot_metric_over_time(
    metric_values=ensemble_memory_kb,
    title="Ensemble Traced Memory Usage over Time",
    ylabel="Memory (KB)",
)